# Week 3 — PySpark for Store-Level Insights
### Retail Sales Performance Dashboard Capstone

In [10]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName('RetailSalesWeek3') \
    .master('local[*]') \
    .getOrCreate()

print('Spark version:', spark.version)


Spark version: 4.1.2


## Load cleaned sales data (from Week 2 output)

In [11]:
df = spark.read.csv('sales_cleaned.csv', header=True, inferSchema=True)
df.printSchema()
df.show(10)


root
 |-- sale_id: integer (nullable = true)
 |-- store_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- price: double (nullable = true)
 |-- cost: double (nullable = true)
 |-- sale_date: date (nullable = true)
 |-- discount_pct: double (nullable = true)
 |-- revenue: double (nullable = true)
 |-- total_cost: double (nullable = true)
 |-- profit: double (nullable = true)
 |-- profit_margin_pct: double (nullable = true)

+-------+--------+----------+--------+------+------+----------+------------+-------+----------+-------+-----------------+
|sale_id|store_id|product_id|quantity| price|  cost| sale_date|discount_pct|revenue|total_cost| profit|profit_margin_pct|
+-------+--------+----------+--------+------+------+----------+------------+-------+----------+-------+-----------------+
|      1|       1|         1|       5| 599.0| 350.0|2026-06-01|         5.0|2845.25|    1750.0|1095.25|            38.49|
|      2|       1

## Add month column for monthly aggregation

In [12]:
from pyspark.sql.functions import col, month, year, concat_ws, lpad

df = df.withColumn('sale_month', concat_ws('-', year(col('sale_date')), lpad(month(col('sale_date')), 2, '0')))
df.select('sale_id', 'store_id', 'sale_date', 'sale_month', 'revenue').show(10)


+-------+--------+----------+----------+-------+
|sale_id|store_id| sale_date|sale_month|revenue|
+-------+--------+----------+----------+-------+
|      1|       1|2026-06-01|   2026-06|2845.25|
|      2|       1|2026-06-01|   2026-06| 4990.0|
|      3|       2|2026-06-02|   2026-06| 4047.3|
|      4|       2|2026-06-02|   2026-06|14995.0|
|      5|       3|2026-06-03|   2026-06|2974.15|
|      6|       3|2026-06-03|   2026-06| 6293.0|
|      7|       4|2026-06-04|   2026-06| 4936.2|
|      8|       4|2026-06-04|   2026-06| 4794.0|
|      9|       5|2026-06-05|   2026-06| 4792.0|
|     10|       5|2026-06-05|   2026-06| 4790.4|
+-------+--------+----------+----------+-------+
only showing top 10 rows


## Filter underperforming products
Defined as: products with total quantity sold below the overall average, or with average profit margin below 35%.

In [13]:
from pyspark.sql.functions import sum as spark_sum, avg, round as spark_round, count

product_summary = df.groupBy('product_id').agg(
    spark_sum('quantity').alias('total_quantity'),
    spark_sum('revenue').alias('total_revenue'),
    spark_round(avg('profit_margin_pct'), 2).alias('avg_profit_margin'),
    count('sale_id').alias('num_sales')
)

avg_quantity_overall = product_summary.agg(avg('total_quantity')).collect()[0][0]
print('Average quantity sold per product:', round(avg_quantity_overall, 2))

underperforming = product_summary.filter(
    (col('total_quantity') < avg_quantity_overall) | (col('avg_profit_margin') < 35)
).orderBy('total_revenue')

underperforming.show()


Average quantity sold per product: 15.12
+----------+--------------+-------------+-----------------+---------+
|product_id|total_quantity|total_revenue|avg_profit_margin|num_sales|
+----------+--------------+-------------+-----------------+---------+
|         8|            14|      11186.0|            49.94|        3|
|         2|             8|      11242.5|            35.51|        3|
|         7|            12|     15133.35|            44.22|        3|
|         5|             6|     16620.25|            24.11|        3|
|         4|             9|      26991.0|            39.98|        3|
+----------+--------------+-------------+-----------------+---------+



## Group by store — average monthly revenue

In [14]:
store_monthly = df.groupBy('store_id', 'sale_month').agg(
    spark_sum('revenue').alias('monthly_revenue'),
    spark_sum('profit').alias('monthly_profit')
)

store_avg_monthly = store_monthly.groupBy('store_id').agg(
    spark_round(avg('monthly_revenue'), 2).alias('avg_monthly_revenue'),
    spark_round(avg('monthly_profit'), 2).alias('avg_monthly_profit')
).orderBy(col('avg_monthly_revenue').desc())

store_avg_monthly.show()


+--------+-------------------+------------------+
|store_id|avg_monthly_revenue|avg_monthly_profit|
+--------+-------------------+------------------+
|       4|           29404.65|          13054.65|
|       5|            29272.6|            9972.6|
|       2|           29128.15|          11028.15|
|       3|           20254.15|           9204.15|
|       1|           16230.25|           7680.25|
+--------+-------------------+------------------+



## Identify underperforming stores
Stores with average monthly revenue below the overall average across all stores.

In [15]:
avg_revenue_overall = store_avg_monthly.agg(avg('avg_monthly_revenue')).collect()[0][0]
print('Overall average monthly revenue across stores:', round(avg_revenue_overall, 2))

underperforming_stores = store_avg_monthly.filter(col('avg_monthly_revenue') < avg_revenue_overall)
underperforming_stores.show()


Overall average monthly revenue across stores: 24857.96
+--------+-------------------+------------------+
|store_id|avg_monthly_revenue|avg_monthly_profit|
+--------+-------------------+------------------+
|       3|           20254.15|           9204.15|
|       1|           16230.25|           7680.25|
+--------+-------------------+------------------+



## Save outputs — underperforming products/store summary

In [18]:
underperforming.toPandas().to_csv('output_underperforming_products.csv', index=False)
store_avg_monthly.toPandas().to_csv('output_store_monthly_summary.csv', index=False)

print('Saved underperforming_products and store_monthly_summary as CSV')

Saved underperforming_products and store_monthly_summary as CSV


## Full Store + Product Performance Report

In [19]:
full_report = df.groupBy('store_id', 'product_id').agg(
    spark_sum('quantity').alias('total_quantity'),
    spark_sum('revenue').alias('total_revenue'),
    spark_round(avg('profit_margin_pct'), 2).alias('avg_profit_margin')
).orderBy('store_id', col('total_revenue').desc())

full_report.show(50)


+--------+----------+--------------+-------------+-----------------+
|store_id|product_id|total_quantity|total_revenue|avg_profit_margin|
+--------+----------+--------------+-------------+-----------------+
|       1|         4|             2|       5998.0|            39.98|
|       1|         3|            10|       4990.0|            59.92|
|       1|         1|             5|      2845.25|            38.49|
|       1|         8|             3|       2397.0|            49.94|
|       2|         4|             5|      14995.0|            39.98|
|       2|         2|             3|       4047.3|            33.29|
|       2|         7|             3|      3702.15|            43.28|
|       2|         1|             6|       3234.6|            35.08|
|       2|         5|             1|       3149.1|            30.14|
|       3|         6|             7|       6293.0|            49.94|
|       3|         2|             3|       4497.0|            39.96|
|       3|         8|             

In [20]:
spark.stop()
